# 連合学習チュートリアル（動作確認・時間計測用）

方針②（`run_simulation` 直呼び・CPU実行）の最小構成です。
教材として整える前の素の状態で、**実行時間を測る**ことが目的です。

**使い方**: 上のセルから順に実行してください（メニューの「ランタイム > すべてのセルを実行」でもOK）。


## 1. ライブラリのインストール
Colab では毎回必要です。3〜5分程度かかります。

In [ ]:
!pip install -q "flwr[simulation]>=1.18.0" "flwr-datasets[vision]>=0.5.0" torch torchvision

## 2. モデル・データ処理の定義（task の中身）
元コードの `task.py` 相当です。MNIST と Fashion-MNIST を切り替えられるようにし、
データセットIDを正式名（`ylecun/mnist` など）に修正しています。

In [ ]:
from collections import OrderedDict
import torch
import torch.nn as nn
import torch.nn.functional as F
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner
from torch.utils.data import DataLoader
from torchvision.transforms import Compose, Normalize, ToTensor

# dataset registry: short name -> (HF id, normalization)
DATASETS = {
    "mnist": ("ylecun/mnist", (0.1307,), (0.3081,)),
    "fashion_mnist": ("zalando-datasets/fashion_mnist", (0.2860,), (0.3530,)),
}

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32); self.pool1 = nn.MaxPool2d(2, 2); self.dropout1 = nn.Dropout(0.25)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64); self.pool2 = nn.MaxPool2d(2, 2); self.dropout2 = nn.Dropout(0.25)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 7 * 7, 128); self.dropout3 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x)))); x = self.dropout1(x)
        x = self.pool2(F.relu(self.bn2(self.conv2(x)))); x = self.dropout2(x)
        x = self.flatten(x); x = F.relu(self.fc1(x)); x = self.dropout3(x)
        return self.fc2(x)

fds = None
_current_dataset = None

def load_data(partition_id, num_partitions, dataset_name="mnist", random_seed=42):
    global fds, _current_dataset
    hf_id, mean, std = DATASETS[dataset_name]
    if fds is None or _current_dataset != dataset_name:
        partitioner = IidPartitioner(num_partitions=num_partitions)
        fds = FederatedDataset(dataset=hf_id, partitioners={"train": partitioner})
        _current_dataset = dataset_name
    partition = fds.load_partition(partition_id, "train")
    ptt = partition.train_test_split(test_size=0.2, seed=random_seed)
    tfm = Compose([ToTensor(), Normalize(mean, std)])
    def apply(b):
        b["image"] = [tfm(img) for img in b["image"]]; return b
    ptt = ptt.with_transform(apply)
    train_ds = ptt["train"]
    drop_last = len(train_ds) >= 32
    trainloader = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=drop_last)
    testloader = DataLoader(ptt["test"], batch_size=32)
    return trainloader, testloader

def train(net, trainloader, epochs, device, learning_rate=0.001):
    net.to(device)
    opt = torch.optim.Adam(net.parameters(), lr=learning_rate)
    crit = torch.nn.CrossEntropyLoss().to(device)
    net.train()
    losses = []
    for _ in range(epochs):
        rl, nb = 0.0, 0
        for batch in trainloader:
            images = batch["image"].to(device); labels = batch["label"].to(device)
            opt.zero_grad(); out = net(images); loss = crit(out, labels)
            loss.backward(); opt.step(); rl += loss.item(); nb += 1
        losses.append(rl / nb if nb else 0.0)
    return sum(losses) / len(losses) if losses else 0.0

def test(net, testloader, device):
    net.to(device)
    crit = torch.nn.CrossEntropyLoss()
    correct, loss_sum, total = 0, 0.0, 0
    net.eval()
    with torch.no_grad():
        for batch in testloader:
            images = batch["image"].to(device); labels = batch["label"].to(device)
            out = net(images)
            loss_sum += crit(out, labels).item() * len(labels); total += len(labels)
            correct += (torch.max(out.data, 1)[1] == labels).sum().item()
    return (loss_sum/total if total else 0.0), (correct/total if total else 0.0)

def get_weights(net):
    return [val.cpu().numpy() for _, val in net.state_dict().items()]

def set_weights(net, parameters):
    params_dict = zip(net.state_dict().keys(), parameters)
    state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
    net.load_state_dict(state_dict, strict=True)


## 3. 連合学習の実行関数（run_federated）
`run_simulation` をノートブックから直接呼びます。結果は `history` 辞書に入ります。

In [ ]:
import torch
from flwr.client import ClientApp, NumPyClient
from flwr.server import ServerApp, ServerAppComponents, ServerConfig
from flwr.server.strategy import FedAvg
from flwr.simulation import run_simulation
from flwr.common import Context, ndarrays_to_parameters
# （task の各関数は上のセルで定義済み）

# history collector (in-memory, no Excel)
history = {"round": [], "train_loss": [], "eval_loss": [], "accuracy": []}

def run_federated(dataset_name="mnist", num_clients=5, num_rounds=3,
                  local_epochs=1, learning_rate=0.001, fraction_fit=1.0, seed=42):
    for k in history: history[k].clear()
    DEVICE = torch.device("cpu")

    class FlowerClient(NumPyClient):
        def __init__(self, net, trainloader, valloader, local_epochs):
            self.net=net; self.trainloader=trainloader; self.valloader=valloader
            self.local_epochs=local_epochs; self.device=DEVICE; self.net.to(DEVICE)
        def fit(self, parameters, config):
            set_weights(self.net, parameters)
            lr=float(config.get("learning_rate", learning_rate))
            avg=train(self.net, self.trainloader, self.local_epochs, self.device, learning_rate=lr)
            return get_weights(self.net), len(self.trainloader.dataset), {"train_loss":avg}
        def evaluate(self, parameters, config):
            set_weights(self.net, parameters)
            loss, acc = test(self.net, self.valloader, self.device)
            return loss, len(self.valloader.dataset), {"accuracy":acc, "loss":loss}

    def client_fn(context: Context):
        pid=context.node_config["partition-id"]; npart=context.node_config["num-partitions"]
        tl, vl = load_data(pid, npart, dataset_name=dataset_name, random_seed=seed)
        return FlowerClient(Net(), tl, vl, local_epochs).to_client()

    client_app = ClientApp(client_fn=client_fn)

    def weighted_average(metrics):
        tot=sum(n for n,_ in metrics)
        if tot==0: return {"accuracy":0.0,"loss":0.0}
        acc=sum(n*m.get("accuracy",0) for n,m in metrics)/tot
        loss=sum(n*m.get("loss",0) for n,m in metrics)/tot
        return {"accuracy":acc,"loss":loss}
    def fit_agg(metrics):
        tot=sum(n for n,_ in metrics)
        return {"train_loss": sum(n*m.get("train_loss",0) for n,m in metrics)/tot if tot else 0.0}

    class TrackingFedAvg(FedAvg):
        def aggregate_fit(self, rnd, results, failures):
            p,m=super().aggregate_fit(rnd,results,failures)
            self._last_train_loss=m.get("train_loss") if m else None
            return p,m
        def aggregate_evaluate(self, rnd, results, failures):
            loss,m=super().aggregate_evaluate(rnd,results,failures)
            history["round"].append(rnd)
            history["train_loss"].append(getattr(self,"_last_train_loss",None))
            history["eval_loss"].append(m.get("loss") if m else None)
            history["accuracy"].append(m.get("accuracy") if m else None)
            print(f"  [round {rnd}] train_loss={getattr(self,'_last_train_loss',None):.4f}  eval_loss={m.get('loss'):.4f}  accuracy={m.get('accuracy'):.4f}")
            return loss,m

    def server_fn(context: Context):
        params=ndarrays_to_parameters(get_weights(Net()))
        strat=TrackingFedAvg(
            fraction_fit=fraction_fit, fraction_evaluate=1.0,
            min_available_clients=num_clients,
            min_fit_clients=max(1,int(num_clients*fraction_fit)),
            min_evaluate_clients=num_clients,
            initial_parameters=params,
            evaluate_metrics_aggregation_fn=weighted_average,
            fit_metrics_aggregation_fn=fit_agg,
            on_fit_config_fn=lambda r: {"learning_rate":learning_rate},
        )
        return ServerAppComponents(strategy=strat, config=ServerConfig(num_rounds=num_rounds))

    server_app = ServerApp(server_fn=server_fn)
    run_simulation(
        server_app=server_app, client_app=client_app,
        num_supernodes=num_clients,
        backend_config={"client_resources":{"num_cpus":1,"num_gpus":0.0}},
    )
    return dict(history)


## 4. 動作確認＆時間計測
まず軽い設定で1回流して、最後まで通るか・何秒かかるかを見ます。
`%%time` でセルの実行時間が表示されます。

In [ ]:
%%time
h = run_federated(dataset_name="mnist", num_clients=5, num_rounds=3,
                  local_epochs=1, learning_rate=0.001, seed=42)
print("最終精度:", h["accuracy"][-1])

## 5. 精度の推移をグラフ表示
（教材ではここを各自の画面で見せる想定）

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.plot(h["round"], h["accuracy"], marker="o", label="accuracy")
plt.xlabel("round"); plt.ylabel("accuracy"); plt.ylim(0,1)
plt.title("Federated Learning: accuracy per round")
plt.grid(True); plt.legend(); plt.show()

## 6. 別設定や別データセットで時間を測る（任意）
ラウンド数・クライアント数・データセットを変えて、それぞれ何秒かかるか測ってみてください。

In [ ]:
%%time
# 例: Fashion-MNIST に切り替え
h2 = run_federated(dataset_name="fashion_mnist", num_clients=5, num_rounds=3)
print("最終精度:", h2["accuracy"][-1])